# 🧪 LAB: Manual MLPs for Classification and Regression

In this lab, you will use `PyTorch` to implement manually a multi-layer perceptron (MLP) for three different tasks: binary classification, multi-class classification and regression. 

## General instructions to complete in ALL three tasks:

1. ***IMPLEMENTATION***:
   
Implement a separate class for each task:

  - `BinaryMLP` for the binary classification task  
  - `MultiClassMLP` for the multi-class classification task  
  - `RegressionMLP` for the regression task

   Each class must include the following methods:

  - `__init__` for initializing 1 or 2 hidden layers. (*Help*: having a second hidden layer only means the need of having a extra set of weights and intercept). 
  - `forward` to transfer information from the input to the output layer. (*Help*: make sure that information passes through a second hidden layer if there is one).
  - `cost` computing the cost.
  - `fit` for training, using the gradients to manually update the parameters of the model. **Use stochastic gradient descent to update your weights**. *N.B.* You may probably reuse much of the code we used in this week tutorial already. 
  - `predict` to convert the information at the output layer into the required output.

2. ***DATA PREPARATION***

For each task, you will be provided with a toy dataset. For each dataset:

  - Split into training and holdout sets (use an 80/20 split)
  - Standardize the features properly, avoiding data leakage
  - Convert all data into `PyTorch` Tensors for compatibility

3. ***MODEL TRAINING AND EVALUATION***

Instantiate the model for each task and train it under different hyperparameter configurations. In each case, record the performance on **both** the training and holdout sets using the appropriate metric for the task (accuracy, MSE, etc.). You should explore the following configurations:

  - One hidden layer, varying the number of hidden units (use ReLU as their activation function)
  - Same number of hidden units across one or two hidden layers (use ReLU as their activation function)
  - Repeat the above setups using Tanh activation instead of ReLU

Present your results in a compact way (e.g., a summary table, a data frame etc).

**NOTE**: When training your model, use a fixed learning rate of your choice (e.g., 0.01 is a reasonable starting point) and a reasonably large number of  epochs (e.g., 100–200) based on how training and holdout performance evolve.

4. ***REFLECTION AND DISCUSSION***

Reflect on the impact of the different hyperparameter settings:

- How does the number of hidden units affect performance?
- What changes when using two layers instead of one?
- How does the activation function (ReLU vs. Tanh) influence results?

Please elaborate your answers.

---

**Collaboration Note**: This assignment is designed to support collaborative work. We encourage you to divide tasks among group members so that everyone can contribute meaningfully. Many components of the assignment can be approached in parallel or split logically across team members. Good coordination and thoughtful integration of your work will lead to a stronger final result.

**Ideally, each group member should be responsible for one of the separate tasks.** BUT, everyone should help each other along the way, both reviewing and refining results and discussion.

---

In total, this lab assignment will be worth **100 points**.

--- 
**Submission notes**:

* Write down all group members' names, or at least the group name (if you have one and you previously provided it), in the first cell of the notebook.

* Verify that the notebook runs as expected and that all required outputs are included.


In [1]:
# NAME(s) = "Shreyans Jain, Luke Hakso, Thomas Gao"

## 1. Pre-implementation Group Discussion (15 points)

Discuss and agree on:

1) What cost function should be used for each of the below task.

All three tasks come down to the same question of what distribution the target follows, and the loss falls out of that once we decide. For binary classification we should use binary cross entropy, since it is the negative log likelihood of a Bernoulli target and therefore the right loss when the output is a probability. It also behaves well in practice, because with a sigmoid output the gradient reduces to (p - y)*x and so does not vanish the way MSE would. Multi-class classification is the same thing with more classes, so we use categorical cross entropy, which with one-hot labels is just minus the log of the probability given to the correct class. The one implementation detail we agreed on is clamping the predicted probabilities away from 0 by a small epsilon so that log(0) can't give NaN. Regression breaks the pattern only in that the target is continuous rather than a label, so we use mean squared error. MSE is the negative log likelihood under Gaussian noise, which is exactly how this data was generated, and it is smooth everywhere.

2) What changes are needed in the output layer for each of these tasks. In particular, consider the number of units and the activation function.

Each task asks for a different kind of answer, so the output layer has to be shaped to produce it. Binary classification needs one output unit with a sigmoid, giving one probability for class 1 while the other class is implied as 1 minus that value. Multi-class classification needs three units, one per class, with a softmax, because softmax forces the outputs to sum to 1 so they form a real distribution, which three independent sigmoids would not. Regression needs one unit with no activation at all, since the target is unbounded and any squashing function would cap what the model can predict. What stood out to us is how little else changes, because the hidden layers are identical in all three cases and only the last layer and the cost function differ. The pairings do matter though, with sigmoid going with binary cross entropy, softmax with categorical cross entropy, and identity with MSE, since mixing them breaks the gradients.

3) Why it is important to standardize the data before training each model.

Standardizing is mostly about making gradient descent behave. Features on different scales turn the loss surface into a long narrow valley, so one learning rate is too large in one direction and too small in another, which matters more here since we are writing SGD by hand with no adaptive optimizer. It also keeps inputs near zero where Tanh and sigmoid still have usable gradients instead of saturating. We fit the scaler on the training split only and apply it to the holdout, since fitting on everything leaks holdout information and flatters the results. For regression we standardize y too, because its raw values run into the hundreds and the gradients would blow up SGD. That means our regression MSEs are on the standardized scale, where an MSE of 1 is what you'd get by always predicting the mean.

4) How you could detect overfitting when training your models.

The simplest signal is just to track training and holdout loss at every epoch rather than only at the end. Overfitting then appears as training loss still dropping while holdout loss flattens and then turns back upward, and the bottom of the holdout curve marks the best model. What matters is the gap rather than the absolute level, so a widening gap means overfitting while both being bad with a small gap means underfitting instead. Across the hyperparameter configurations we test, we expect this to show up as a trend, with holdout performance improving as we add hidden units up to a point and then getting worse, most likely in the largest two layer model. For the two dimensional classification tasks we can also plot the decision boundary, where fingers reaching out to grab single points mean the model is memorising noise. One caveat we noted is that tuning repeatedly against the same 20 percent holdout overfits that holdout too, so cross-validation or a third untouched split would be the proper fix.


## 2. Binary Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [2]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
import torch

X, y = make_moons(n_samples=1000, noise=0.2, random_state=42)
print(f"X shape: {X.shape}, y shape: {y.shape}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# fit the scaler on the training set only to avoid leakage
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
# column vectors so they line up with the (n, 1) network output
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}, y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

X shape: (1000, 2), y shape: (1000,)
X_train shape: torch.Size([800, 2]), X_test shape: torch.Size([200, 2]), y_train shape: torch.Size([800, 1]), y_test shape: torch.Size([200, 1])


In [ ]:
class BaseMLP:
    """Create a base class because hidden layers, backprop and SGD are the same for every task, so they live here. Subclasses define the output layer."""

    def __init__(self, input_size, hidden_size, output_size, n_hidden_layers=1, activation="relu", seed=0):
        torch.manual_seed(seed)
        self.activation = activation
        sizes = [input_size] + [hidden_size] * n_hidden_layers + [output_size]
        # small random weights, zero intercepts; one (W, b) pair per layer
        self.weights = [torch.randn(sizes[i], sizes[i + 1]) * np.sqrt(1 / sizes[i]) for i in range(len(sizes) - 1)]
        self.biases = [torch.zeros(sizes[i + 1]) for i in range(len(sizes) - 1)]

    # --- implemented by each subclass ---
    def output_activation(self, z):
        raise NotImplementedError

    def cost(self, y_hat, y):
        raise NotImplementedError

    def predict(self, X):
        raise NotImplementedError

    def score(self, X, y):
        raise NotImplementedError

    # --- shared ---
    def act(self, z):
        return torch.relu(z) if self.activation == "relu" else torch.tanh(z)

    def act_derivative(self, z, a):
        return (z > 0).float() if self.activation == "relu" else 1 - a ** 2

    def forward(self, X):
        self.zs, self.activations = [], [X]
        a = X

        # zip weights and biases so we get them in pairs, loop so it works with different numbers of hidden layers
        for W, b in zip(self.weights[:-1], self.biases[:-1]):
            z = a @ W + b
            a = self.act(z)
            self.zs.append(z)
            self.activations.append(a)
        z = a @ self.weights[-1] + self.biases[-1] # grab last w, b
        a = self.output_activation(z)
        self.zs.append(z)
        self.activations.append(a)
        return a

    def backward(self, y, learning_rate):
        m = y.shape[0] # batch size
        # sigmoid + binary cross-entropy, softmax + categorical cross-entropy and
        # identity + 1/2 MSE all reduce to (prediction - target) at the output layer
        dZ = self.activations[-1] - y
        for i in reversed(range(len(self.weights))):
            dW = self.activations[i].T @ dZ / m
            db = dZ.sum(dim=0) / m
            if i > 0:
                dZ = (dZ @ self.weights[i].T) * self.act_derivative(self.zs[i - 1], self.activations[i])
            self.weights[i] -= learning_rate * dW
            self.biases[i] -= learning_rate * db

    def fit(self, X, y, epochs=200, learning_rate=0.01, batch_size=1, X_val=None, y_val=None):
        self.history = []
        n = X.shape[0]
        for epoch in range(epochs):
            # stochastic gradient descent: shuffle, then update on each mini-batch
            perm = torch.randperm(n)
            for start in range(0, n, batch_size):
                idx = perm[start:start + batch_size]
                self.forward(X[idx])
                self.backward(y[idx], learning_rate)
            record = {"epoch": epoch + 1, "train_cost": self.cost(self.forward(X), y).item()}
            if X_val is not None:
                record["val_cost"] = self.cost(self.forward(X_val), y_val).item()
            self.history.append(record)
        return self

    def predict_proba(self, X):
        return self.forward(X)

In [ ]:
class BinaryMLP(BaseMLP):
    metric = "acc"

    def __init__(self, input_size, hidden_size, n_hidden_layers=1, activation="relu", seed=0):
        # one output unit holding P(y = 1)
        super().__init__(input_size, hidden_size, 1, n_hidden_layers, activation, seed)

    def output_activation(self, z):
        return 1 / (1 + torch.exp(-z))

    def cost(self, y_hat, y):
        # binary cross-entropy; clamp so log(0) can't happen
        y_hat = torch.clamp(y_hat, 1e-7, 1 - 1e-7)
        return -torch.mean(y * torch.log(y_hat) + (1 - y) * torch.log(1 - y_hat))

    def predict(self, X):
        return (self.forward(X) > 0.5).float()

    def score(self, X, y):
        # accuracy
        return (self.predict(X) == y).float().mean().item()

In [5]:
mlp = BinaryMLP(input_size=2, hidden_size=10)
mlp.fit(X_train, y_train, epochs=200, learning_rate=0.01)
print(f"Train accuracy: {mlp.score(X_train, y_train):.3f}")
print(f"Holdout accuracy: {mlp.score(X_test, y_test):.3f}")

Train accuracy: 0.969
Holdout accuracy: 0.990


In [6]:
configs = (
    # one hidden layer, varying the number of units
    [(h, 1) for h in [2, 4, 8, 16, 32]]
    # same number of units across one vs. two hidden layers
    + [(16, 2)]
)

def run_sweep(model_cls, X_train, y_train, X_test, y_test, epochs=200, learning_rate=0.01, **model_kwargs):
    results = []
    for activation in ["relu", "tanh"]:
        for hidden_size, n_layers in configs:
            model = model_cls(input_size=X_train.shape[1], hidden_size=hidden_size,
                              n_hidden_layers=n_layers, activation=activation, **model_kwargs)
            model.fit(X_train, y_train, epochs=epochs, learning_rate=learning_rate)
            results.append({
                "activation": activation,
                "hidden_layers": n_layers,
                "hidden_units": hidden_size,
                f"train_{model.metric}": round(model.score(X_train, y_train), 3),
                f"holdout_{model.metric}": round(model.score(X_test, y_test), 3),
            })
    return pd.DataFrame(results)

binary_results = run_sweep(BinaryMLP, X_train, y_train, X_test, y_test)
binary_results

,activation,hidden_layers,hidden_units,train_acc,holdout_acc
0,relu,1,2,0.870,0.860
1,relu,1,4,0.978,0.990
2,relu,1,8,0.974,0.990
3,relu,1,16,0.978,0.995
4,relu,1,32,0.976,0.985
5,relu,2,16,0.978,0.990
6,tanh,1,2,0.865,0.835
7,tanh,1,4,0.973,0.970
8,tanh,1,8,0.975,0.995
9,tanh,1,16,0.974,0.995


### Binary Classification Reflection

**Number of hidden units.** Capacity matters a lot at the low end and not much after that. With only 2 hidden units both activations underfit, with ReLU at 0.870 train and 0.860 holdout and Tanh at 0.865 and 0.835, since two units can only bend the decision boundary in a couple of places and that is not enough to trace the two interlocking moons. Going to 4 units is the big jump, taking ReLU to 0.990 holdout. From 4 to 32 units train accuracy stays between 0.973 and 0.978 for both activations, and holdout stays between 0.970 and 0.995. We think that plateau comes from the data rather than the model. With noise set to 0.2 some points from each moon can end up on the wrong side. We also do not see overfitting even at 32 units, because train accuracy never climbs towards 1.0.

**One versus two hidden layers.** With 16 units, adding a second layer barely changes anything. ReLU goes from 0.978 and 0.995 to 0.978 and 0.990, and Tanh goes from 0.974 and 0.995 to 0.979 and 0.985. Train accuracy goes up a little with Tanh while holdout drops by one or two test points, which is the direction we would expect if the extra capacity is being spent on noise, but the effect is too small to really call. The moons boundary is simple enough that one wide layer already captures it, so depth just adds parameters and training time without giving the task anything it needs.

**ReLU versus Tanh.** Once there are enough units the two are basically interchangeable, and both reach 0.995 holdout at their best. The difference only shows up when capacity is tight. ReLU builds the boundary out of straight segments joined at kinks, while Tanh builds it out of smooth S shaped pieces. At 4 units ReLU did better with 0.990 holdout against 0.970 for Tanh, while at 8 units Tanh was slightly ahead, so neither one dominates. At 2 units Tanh did a bit worse on holdout at 0.835 compared to 0.860, but the two had almost the same train accuracy at 0.865 and 0.870, so that gap is five holdout points and probably noise rather than a real difference between the activations.

## 2. Multi-Class Classification (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

---
**NOTE**: This will likely be the most challenging exercise. To help you, here are some pointers:

 - The **output (last) layer** should have as many units as there are classes in your data.  

 - The **activation function of the output layer must be softmax**, not sigmoid. Softmax ensures that all output values are between 0 and 1 and sum to 1, so they can be interpreted as probabilities across the different classes. (*Help*: you may want to look into `torch.nn.functional` to find its implementation...)

 - For a multi-class problem, your target variable `y` should be **one-hot encoded**. For example:  
   - Label = 0 → [1, 0, 0]  
   - Label = 1 → [0, 1, 0]  
   - Label = 2 → [0, 0, 1]  
   You can easily achieve this with `OneHotEncoder` from `sklearn.preprocessing`.  

 - The **predicted class** corresponds to the unit with the highest probability.  
   Example:  
   - `[0.1, 0.3, 0.6] → class 2`  
   - `[0.6, 0.2, 0.2] → class 0`  

 - For this exercise, you need to **implement the categorical cross-entropy loss** (an extension of binary cross-entropy to multiple classes). It is defined as:  

   $$\sum_{c=1}^{l} y_{o,c}\,\log(p_{o,c}),$$

   where $l$ is the number of classes, $y_{o,c}$ is the one-hot encoded label for observation $o$, and $p_{o,c}$ is the predicted probability for class $c$ (after applying softmax). The log is the natural logarithm.

---

In [7]:
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_classes=3, 
                           n_clusters_per_class=1, 
                           n_features=2, n_informative=2, n_redundant=0, random_state=1234, flip_y=0.15)

In [8]:
from sklearn.preprocessing import OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# fit the scaler on the training set only to avoid leakage
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)

# one-hot targets: label 2 -> [0, 0, 1]
encoder = OneHotEncoder(sparse_output=False)
y_train = torch.tensor(encoder.fit_transform(y_train.reshape(-1, 1)), dtype=torch.float32)
y_test = torch.tensor(encoder.transform(y_test.reshape(-1, 1)), dtype=torch.float32)
n_classes = y_train.shape[1]
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}, y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

X_train shape: torch.Size([800, 2]), X_test shape: torch.Size([200, 2]), y_train shape: torch.Size([800, 3]), y_test shape: torch.Size([200, 3])


In [9]:
import torch.nn.functional as F

class MultiClassMLP(BaseMLP):
    metric = "acc"

    def __init__(self, input_size, hidden_size, n_classes, n_hidden_layers=1, activation="relu", seed=0):
        # one output unit per class
        super().__init__(input_size, hidden_size, n_classes, n_hidden_layers, activation, seed)

    def output_activation(self, z):
        # each row becomes a probability distribution over the classes
        return F.softmax(z, dim=1)

    def cost(self, y_hat, y):
        # categorical cross-entropy, averaged over observations
        y_hat = torch.clamp(y_hat, 1e-7, 1.0)
        return -torch.mean(torch.sum(y * torch.log(y_hat), dim=1))

    def predict(self, X):
        # class with the highest probability
        return torch.argmax(self.forward(X), dim=1)

    def score(self, X, y):
        # accuracy; y is one-hot, so recover the labels first
        return (self.predict(X) == y.argmax(dim=1)).float().mean().item()

In [10]:
mlp = MultiClassMLP(input_size=2, hidden_size=10, n_classes=n_classes)
mlp.fit(X_train, y_train, epochs=200, learning_rate=0.01)
print(f"Train accuracy: {mlp.score(X_train, y_train):.3f}")
print(f"Holdout accuracy: {mlp.score(X_test, y_test):.3f}")

Train accuracy: 0.820
Holdout accuracy: 0.800


In [11]:
multiclass_results = run_sweep(MultiClassMLP, X_train, y_train, X_test, y_test, n_classes=n_classes)
multiclass_results

,activation,hidden_layers,hidden_units,train_acc,holdout_acc
0,relu,1,2,0.795,0.760
1,relu,1,4,0.817,0.800
2,relu,1,8,0.817,0.790
3,relu,1,16,0.822,0.790
4,relu,1,32,0.812,0.795
5,relu,2,16,0.830,0.800
6,tanh,1,2,0.801,0.790
7,tanh,1,4,0.822,0.785
8,tanh,1,8,0.821,0.790
9,tanh,1,16,0.824,0.800


### Multi-Class Classification Reflection

**Number of hidden units.** Every configuration lands between 0.795 and 0.830 train and 0.760 and 0.805 holdout, which tells us this task is limited by the data rather than model size. The dataset was generated with `flip_y` set to 0.15, so 15 percent of labels were reassigned at random. Around 10 percent actually end up wrong, since a third of those reassignments land back on the correct class by chance, and the three clusters also overlap. No model can predict randomly flipped labels, so accuracy in the low 0.80s is close to the ceiling. The only clear effect of hidden units is at the bottom end, where ReLU with 2 units is the weakest model at 0.760 holdout. From 4 units onwards results move by a point or two in no consistent direction.

**One versus two hidden layers.** The two layer ReLU model has the highest train accuracy of the whole sweep at 0.830 with 0.800 holdout, and the two layer Tanh model gets 0.825 and 0.795. Compared to their 16 unit single layer versions, the second layer doesn't give much benefit. ReLU gains about a point on both train and holdout, going from 0.822 and 0.790 to 0.830 and 0.800, which is two holdout observations. Tanh stays flat on train and loses one observation on holdout, so its gap between train and holdout widens from 0.024 to 0.030. The class boundaries here are close to straight lines, so there isn't much complex structure for the extra depth to find.

**ReLU versus Tanh.** Results are almost identical from 4 units up. Tanh beats ReLU with 0.790 holdout against 0.760 at 2 units. This is probably because when ReLU units "die" and zero out, it hurts more when there are fewer total neurons. Tanh units are never fully switched off, so they keep contributing a usable signal everywhere. Once there are extra units this disadvantage goes away.

## 3: Regression Task (25 points)

Use the dataset below to complete points 1 to 4 in the general instructions for this task.

Use as many cells as needed.

In [12]:
from sklearn.datasets import make_regression
X, y = make_regression(n_samples=1000, n_features=2, n_informative=2, random_state=1234, noise=75)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# fit both scalers on the training set only to avoid leakage; y is scaled too
# because its raw values are in the hundreds and would blow up SGD
x_scaler = StandardScaler()
X_train = torch.tensor(x_scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(x_scaler.transform(X_test), dtype=torch.float32)

y_scaler = StandardScaler()
y_train = torch.tensor(y_scaler.fit_transform(y_train.reshape(-1, 1)), dtype=torch.float32)
y_test = torch.tensor(y_scaler.transform(y_test.reshape(-1, 1)), dtype=torch.float32)
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}, y_train shape: {y_train.shape}, y_test shape: {y_test.shape}")

X_train shape: torch.Size([800, 2]), X_test shape: torch.Size([200, 2]), y_train shape: torch.Size([800, 1]), y_test shape: torch.Size([200, 1])


In [14]:
class RegressionMLP(BaseMLP):
    metric = "mse"

    def __init__(self, input_size, hidden_size, n_hidden_layers=1, activation="relu", seed=0):
        # one linear output unit
        super().__init__(input_size, hidden_size, 1, n_hidden_layers, activation, seed)

    def output_activation(self, z):
        # identity: the output can take any real value
        return z

    def cost(self, y_hat, y):
        # 1/2 MSE so the output-layer gradient is exactly (y_hat - y)
        return 0.5 * torch.mean((y_hat - y) ** 2)

    def predict(self, X):
        return self.forward(X)

    def score(self, X, y):
        # MSE (on standardized y, so 1 - MSE is roughly R^2)
        return torch.mean((self.predict(X) - y) ** 2).item()

In [15]:
from sklearn.linear_model import LinearRegression

mlp = RegressionMLP(input_size=2, hidden_size=10)
mlp.fit(X_train, y_train, epochs=200, learning_rate=0.01)
print(f"Train MSE: {mlp.score(X_train, y_train):.3f}")
print(f"Holdout MSE: {mlp.score(X_test, y_test):.3f}")

# make_regression is linear plus noise, so a linear model is the natural baseline
lin = LinearRegression().fit(X_train, y_train)
print(f"Linear regression holdout MSE: {np.mean((lin.predict(X_test) - y_test.numpy()) ** 2):.3f}")

Train MSE: 0.497
Holdout MSE: 0.524
Linear regression holdout MSE: 0.494


In [16]:
regression_results = run_sweep(RegressionMLP, X_train, y_train, X_test, y_test)
regression_results

,activation,hidden_layers,hidden_units,train_mse,holdout_mse
0,relu,1,2,0.531,0.521
1,relu,1,4,0.493,0.505
2,relu,1,8,0.490,0.507
3,relu,1,16,0.505,0.525
4,relu,1,32,0.478,0.494
5,relu,2,16,0.476,0.503
6,tanh,1,2,0.498,0.499
7,tanh,1,4,0.493,0.501
8,tanh,1,8,0.488,0.499
9,tanh,1,16,0.525,0.566


### Regression Reflection

**Context.** `make_regression` makes a target that is a linear function of the features plus Gaussian noise, with noise set to 75. Since we standardized y, an MSE of about 0.5 means about half the variance in y is noise that no model can explain. The linear regression baseline gets 0.494 holdout MSE, and the best MLP, one layer with 32 ReLU units, ties it at 0.494. None of the MLPs beat it, which is what we expected, because when the true relationship is linear a nonlinear model can at best find the same line.

**Number of hidden units.** Unlike the classification tasks there is no clear trend here. ReLU with 2 units has the worst train fit at 0.531, probably because two ReLU units have a hard time approximating a flat plane. From 4 to 32 units holdout mostly stays between 0.494 and 0.507. The two exceptions are ReLU at 16 units with 0.525 holdout and Tanh at 16 units with 0.566. Both are also worse on train, so we think this is noise from SGD rather than overfitting. With a batch size of 1 and a fixed learning rate the weights keep bouncing around, so the last epoch can land at a slightly worse point.

**One versus two hidden layers.** The two layer ReLU model gets the lowest train MSE in the sweep at 0.476, but its holdout MSE of 0.503 is no better than the linear baseline, so the extra layer is just fitting noise. Two layer Tanh is similar at 0.485 train and 0.496 holdout. There isn't any nonlinear structure in the target for the extra depth to find.

**ReLU versus Tanh.** Tanh does slightly better at small sizes, with 2 to 8 units all between 0.499 and 0.501 holdout compared to 0.505 to 0.521 for ReLU. This is probably because Tanh is almost linear around zero, so a small Tanh network can fit a straight line pretty easily, while ReLU has to build it out of kinked pieces. With enough units the two are about the same, and the differences are small enough that they could just be SGD noise.

## 4. Discussion (5 points)

You created a separate class for each task and likely repeated much of the same code across implementations.

Discuss within your group how could you have leveraged inheritance to make your code more reusable and avoid duplication. Provide examples. Be specific.

We actually built our solution around inheritance from the start, so the clearest answer is to explain what we put in the base class, why it works, and what we would still improve.

Everything that does not depend on the task lives in `BaseMLP`. The `__init__` method builds a list of layer sizes from the input size, the hidden size repeated once per hidden layer, and the output size, then creates one weight matrix and one bias vector for each pair of neighbouring sizes. Because it loops over that list, the same code handles one hidden layer, two, or ten, so we never had to write separate `W1`, `W2` and `W3` variables. The `act` and `act_derivative` methods handle ReLU and Tanh for the hidden layers. The `forward` method loops through the hidden layers and then calls `output_activation` on the last layer, which is the only hook in the forward pass that changes by task. The `backward` and `fit` methods implement backpropagation and SGD once for everything. This works because each of our pairings, sigmoid with binary cross entropy, softmax with categorical cross entropy and identity with half MSE, gives the same output layer gradient of y_hat minus y, and that shared gradient is what lets a single `backward` method stay correct for all three tasks.

Each subclass only overrides what is actually task specific. That means the output size it passes to the parent constructor, the output activation, the cost, `predict`, `score` and a `metric` attribute set to either acc or mse. `BinaryMLP` thresholds at 0.5, `MultiClassMLP` takes the argmax and `RegressionMLP` returns the raw output. `RegressionMLP` ends up at around 20 lines, whereas without the base class each of the three classes would have repeated roughly 60 lines of forward, backward and training code, and any bug fix such as one to the bias gradient would have had to be made three separate times.

The experiments also lean on polymorphism. The `run_sweep` function takes the class itself as an argument and only ever builds the model and uses `fit`, `score` and `metric`. Because every subclass exposes the same interface, one function produces all three results tables, including the right column names.

There are still four things we would improve. First, `BaseMLP` currently raises `NotImplementedError` for the methods subclasses must define, but inheriting from `abc.ABC` and marking those methods with `abstractmethod` would make Python refuse to create an incomplete subclass at all, rather than letting it fail once training has already started. Second, `backward` assumes the canonical output and cost pairing, so if someone wrote a subclass that used sigmoid outputs with MSE it would train with silently wrong gradients. A cleaner design would give the base class an `output_gradient` method that defaults to y_hat minus y, which a subclass could override when its pairing is different. Third, the activations could be small classes instead of if statements, with a ReLU class and a Tanh class that each have their own forward and derivative methods, so adding Leaky ReLU would just mean writing one new class instead of editing the base model. Fourth, `BinaryMLP` and `MultiClassMLP` both compute accuracy the same way, by checking how often the predictions match the labels, so an intermediate `ClassifierMLP` class between `BaseMLP` and the two classifiers could implement `score` once, leaving each subclass to define only how it recovers labels from y.

## 5. Collaboration Reflection (5 points)

As a group, identify **one specific challenge or inefficiency** you encountered while working on this lab. In 3–5 sentences:

1. Briefly describe what happened and how it affected your work.
2. Identify one concrete action your group will take to avoid or better manage this issue in the next lab.

If your group encountered no major problem, identify one aspect of your collaboration that could still be improved.

Our main inefficiency came from splitting the lab one task per person too literally at the start. Each of us began writing a full MLP class for our own task, so we ended up with three versions that initialised weights differently, set up the training loop differently and named things inconsistently, which made it hard to compare results or review each other's code. We only realised how much was duplicated when we tried to combine the notebooks, and refactoring everything into a shared `BaseMLP` took time we could have spent on the experiments and write up. For the next lab we will use our first meeting to agree on a shared interface, meaning a base class skeleton with the method names, input shapes and results table format, and put it in a shared file before anyone starts on their own task.